<a href="https://colab.research.google.com/github/jorgefbarreiros/codes/blob/main/hystheresis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# NK HYSTERESIS MODEL FOR THE U.S. ECONOMY
# ------------------------------------------------------------
# Self-contained Python script for Google Colab / Jupyter
#
# Features:
# - New Keynesian demand block
# - Nominal rigidity via Phillips curve
# - Two-stage productive capacity: physical capital + upstream/intangible capacity
# - Labor heterogeneity: high- and low-productivity workers
# - Hysteresis through:
#     * long-term unemployment
#     * participation scarring
#     * employability scarring
# - Counterfactual models
# - IRFs, decomposition, sensitivity analysis
# - Tables and charts
#
# This is a calibrated research model, fully reproducible and designed
# to be transparent, modifiable, and publication-ready in spirit.
# ============================================================

import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install xlsxwriter
# -----------------------------
# Global plotting configuration
# -----------------------------
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["font.size"] = 10

# -----------------------------
# Output directory
# -----------------------------
OUTPUT_DIR = "nk_hysteresis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 1. CALIBRATION
# ============================================================

def get_us_calibration():
    """
    Calibration targets broadly consistent with recent U.S. macro aggregates.
    Values are quarterly unless otherwise noted.
    """
    pars = {}

    # Preferences and NK core
    pars["beta"] = 0.99                    # quarterly discount factor
    pars["sigma"] = 1.5                    # intertemporal elasticity inverse
    pars["kappa"] = 0.08                   # Phillips curve slope
    pars["phi_pi"] = 1.7                   # Taylor rule inflation coefficient
    pars["phi_y"] = 0.25                   # Taylor rule output coefficient
    pars["rho_i"] = 0.75                   # interest rate smoothing
    pars["r_star_ann"] = 0.02              # 2% annual real neutral rate
    pars["pi_target_ann"] = 0.02           # 2% annual inflation target

    # Demand shock process
    pars["rho_d"] = 0.82                   # persistence of demand disturbance

    # Productive structure
    pars["alpha_k"] = 0.23                 # capital elasticity
    pars["alpha_z"] = 0.12                 # intangible / upstream capacity elasticity
    pars["delta_k"] = 0.025                # depreciation physical capital (quarterly)
    pars["delta_z"] = 0.040                # depreciation intangible/upstream capacity
    pars["inv_share_ss"] = 0.175           # gross private investment / GDP target
    pars["upstream_invest_share"] = 0.030  # part of GDP devoted to Z accumulation
    pars["capital_adj_cost"] = 2.5         # modest adjustment cost curvature
    pars["intangible_adj_cost"] = 3.0      # stronger adjustment for upstream capacity

    # Labor composition
    pars["mass_H"] = 0.60                  # share of high-productivity workers
    pars["mass_L"] = 0.40                  # share of low-productivity workers
    pars["prod_H"] = 1.15                  # productivity of H workers
    pars["prod_L"] = 0.80                  # productivity of L workers
    pars["omega_H"] = 0.58                 # baseline employment share H
    pars["omega_L"] = 0.42                 # baseline employment share L

    # U.S. labor market targets
    pars["u_ss"] = 0.044                   # unemployment rate
    pars["part_ss"] = 0.620                # labor force participation
    pars["ltu_share_ss"] = 0.255           # long-term unemployed share of unemployed
    pars["employment_pop_ss"] = pars["part_ss"] * (1 - pars["u_ss"])

    # Worker flows / hysteresis
    pars["lambda_u"] = 0.18                # flow from short-term to long-term unemployment
    pars["exit_ltu"] = 0.12                # long-term unemployed exit rate (employment + NILF)
    pars["reentry_rate"] = 0.06            # NILF re-entry
    pars["search_success_H"] = 0.45        # baseline hiring success H
    pars["search_success_L"] = 0.38        # baseline hiring success L
    pars["vacancy_sensitivity"] = 1.6      # hiring collapses in downturns
    pars["separation_H"] = 0.018           # job separation H
    pars["separation_L"] = 0.028           # job separation L

    # Hysteresis parameters
    pars["scar_emp_H"] = 0.010             # employability loss from LTU
    pars["scar_emp_L"] = 0.030             # employability loss from LTU
    pars["recovery_emp_H"] = 0.008         # employability recovery when employed
    pars["recovery_emp_L"] = 0.010
    pars["scar_part_H"] = 0.004            # participation loss from LTU/NILF
    pars["scar_part_L"] = 0.015
    pars["recovery_part_H"] = 0.004        # participation recovery
    pars["recovery_part_L"] = 0.006

    # Wage / marginal cost
    pars["real_wage_rigidity"] = 0.75
    pars["mc_sensitivity_y"] = 0.45
    pars["mc_sensitivity_emp"] = 0.30

    # Demand composition
    pars["c_share_ss"] = 1.0 - pars["inv_share_ss"]

    # Welfare penalties
    pars["lambda_pi_welfare"] = 40.0
    pars["lambda_ygap_welfare"] = 4.0
    pars["lambda_u_welfare"] = 15.0

    # Shock scale
    pars["default_demand_shock"] = -0.035  # initial contraction

    # Steady-state policy quantities
    pars["r_star_q"] = (1 + pars["r_star_ann"])**(1/4) - 1
    pars["pi_target_q"] = (1 + pars["pi_target_ann"])**(1/4) - 1

    # Normalize steady-state levels
    pars["Y_ss"] = 1.0
    pars["K_ss"] = 6.8                     # plausible quarterly capital/output ratio
    pars["Z_ss"] = 0.75                    # intangible/upstream capacity stock
    pars["I_ss"] = pars["inv_share_ss"] * pars["Y_ss"]
    pars["Iz_ss"] = pars["upstream_invest_share"] * pars["Y_ss"]

    return pars

# ============================================================
# 2. MODEL CLASS
# ============================================================

class NKHysteresisModel:
    def __init__(
        self,
        pars,
        labor_hysteresis=True,
        capital_hysteresis=True,
        heterogeneity=True,
        name="full_model"
    ):
        self.p = pars.copy()
        self.labor_hysteresis = labor_hysteresis
        self.capital_hysteresis = capital_hysteresis
        self.heterogeneity = heterogeneity
        self.name = name

    # --------------------------------------------------------
    # Steady state
    # --------------------------------------------------------
    def steady_state(self):
        p = self.p

        if self.heterogeneity:
            nH = p["employment_pop_ss"] * p["omega_H"]
            nL = p["employment_pop_ss"] * p["omega_L"]
            u_total = p["part_ss"] - (nH + nL)
            uH = u_total * 0.45
            uL = u_total * 0.55
            ltuH = p["ltu_share_ss"] * uH
            ltuL = p["ltu_share_ss"] * uL
        else:
            nH = p["employment_pop_ss"]
            nL = 0.0
            u_total = p["part_ss"] - nH
            uH = u_total
            uL = 0.0
            ltuH = p["ltu_share_ss"] * uH
            ltuL = 0.0

        ss = {
            "x": 0.0,                       # output gap
            "pi": p["pi_target_q"],
            "i": p["r_star_q"] + p["pi_target_q"],
            "rn": p["r_star_q"],
            "d": 0.0,                       # demand shock state
            "Y": p["Y_ss"],
            "K": p["K_ss"],
            "Z": p["Z_ss"],
            "I": p["I_ss"],
            "Iz": p["Iz_ss"],
            "part_H": p["mass_H"] * p["part_ss"] if self.heterogeneity else p["part_ss"],
            "part_L": p["mass_L"] * p["part_ss"] if self.heterogeneity else 0.0,
            "nH": nH,
            "nL": nL,
            "uH": uH,
            "uL": uL,
            "ltuH": ltuH,
            "ltuL": ltuL,
            "empH": 1.0,
            "empL": 1.0 if self.heterogeneity else 0.0,
            "mc": 0.0,
            "A_eff": 1.0,                   # effective supply capacity
            "N_eff": 1.0,
            "Y_pot": p["Y_ss"],
            "welfare_flow": 0.0,
        }
        return ss

    # --------------------------------------------------------
    # Effective labor and potential output
    # --------------------------------------------------------
    def effective_labor(self, state):
        p = self.p
        if self.heterogeneity:
            labor_eff = (
                p["prod_H"] * state["empH"] * state["nH"] +
                p["prod_L"] * state["empL"] * state["nL"]
            )
        else:
            labor_eff = p["prod_H"] * state["empH"] * state["nH"]
        return max(labor_eff, 1e-8)

    def potential_output(self, state):
        p = self.p
        labor_eff = self.effective_labor(state)
        ypot = (state["K"] ** p["alpha_k"]) * (state["Z"] ** p["alpha_z"]) * (labor_eff ** (1 - p["alpha_k"] - p["alpha_z"]))
        return max(ypot, 1e-8)

    # --------------------------------------------------------
    # One simulation step
    # --------------------------------------------------------
    def step(self, state, d_innov=0.0):
        p = self.p
        s = state.copy()

        # 1) Demand shock process
        d_next = p["rho_d"] * s["d"] + d_innov

        # 2) Potential output from supply side
        y_pot = self.potential_output(s)

        # 3) Natural rate (reduced form)
        rn = p["r_star_q"] + 0.15 * d_next

        # 4) Monetary policy (Taylor rule)
        i_notional = (
            p["rho_i"] * s["i"]
            + (1 - p["rho_i"]) * (
                p["r_star_q"] + p["pi_target_q"]
                + p["phi_pi"] * (s["pi"] - p["pi_target_q"])
                + p["phi_y"] * s["x"]
            )
        )
        i_next = max(i_notional, -0.0025)  # mild lower bound

        # 5) IS curve (reduced form)
        x_next = (
            0.72 * s["x"]
            - (1 / p["sigma"]) * (i_next - s["pi"] - rn)
            + d_next
        )

        # 6) Marginal cost proxy
        employment_gap = (self.effective_labor(s) / max(p["employment_pop_ss"], 1e-8)) - 1.0
        mc_next = p["mc_sensitivity_y"] * x_next + p["mc_sensitivity_emp"] * employment_gap

        # 7) Phillips curve
        pi_next = (
            0.60 * s["pi"]
            + 0.35 * p["pi_target_q"]
            + p["kappa"] * mc_next
        )

        # 8) Actual output
        Y_next = y_pot * (1 + x_next)
        Y_next = max(0.65, Y_next)

        # 9) Demand composition and investment
        # Investment is procyclical and falls sharply in demand contractions
        inv_gap = 1.9 * x_next + 0.4 * d_next
        I_next = p["I_ss"] * max(0.25, 1 + inv_gap)

        iz_gap = 2.3 * x_next + 0.55 * d_next
        Iz_next = p["Iz_ss"] * max(0.15, 1 + iz_gap)

        if not self.capital_hysteresis:
            K_next = p["K_ss"]
            Z_next = p["Z_ss"]
        else:
            # Adjustment cost dampens overreaction
            K_next = (1 - p["delta_k"]) * s["K"] + I_next
            Z_next = (1 - p["delta_z"]) * s["Z"] + Iz_next

        # 10) Hiring intensity declines in contractions
        hiring_multiplier = max(0.15, 1 + p["vacancy_sensitivity"] * x_next)

        # Type-specific employment dynamics
        if self.heterogeneity:
            # Participation stocks by type
            partH = max(1e-6, s["part_H"])
            partL = max(1e-6, s["part_L"])

            # Short-term unemployment
            stuH = max(0.0, s["uH"] - s["ltuH"])
            stuL = max(0.0, s["uL"] - s["ltuL"])

            # Job finding rates
            jfH = p["search_success_H"] * hiring_multiplier * s["empH"]
            jfL = p["search_success_L"] * hiring_multiplier * s["empL"]

            jfH = min(max(jfH, 0.01), 0.90)
            jfL = min(max(jfL, 0.01), 0.90)

            # Separations rise in downturns, more for low type
            sepH = p["separation_H"] * (1 - 0.4 * x_next)
            sepL = p["separation_L"] * (1 - 0.7 * x_next)
            sepH = min(max(sepH, 0.005), 0.10)
            sepL = min(max(sepL, 0.008), 0.15)

            # Employment transitions
            hiresH = jfH * (stuH + 0.35 * s["ltuH"])
            hiresL = jfL * (stuL + 0.20 * s["ltuL"])

            sepsH = sepH * s["nH"]
            sepsL = sepL * s["nL"]

            nH_next = max(0.0, s["nH"] + hiresH - sepsH)
            nL_next = max(0.0, s["nL"] + hiresL - sepsL)

            # Unemployment accounting
            uH_pre = max(0.0, partH - nH_next)
            uL_pre = max(0.0, partL - nL_next)

            # LTU evolution
            ltuH_next = (1 - p["exit_ltu"]) * s["ltuH"] + p["lambda_u"] * max(stuH, 0.0)
            ltuL_next = (1 - p["exit_ltu"]) * s["ltuL"] + p["lambda_u"] * max(stuL, 0.0)
            ltuH_next = min(ltuH_next, uH_pre)
            ltuL_next = min(ltuL_next, uL_pre)

            # Employability
            if self.labor_hysteresis:
                empH_next = (
                    s["empH"]
                    + p["recovery_emp_H"] * (1.0 - s["empH"]) * (s["nH"] / max(partH, 1e-8))
                    - p["scar_emp_H"] * (s["ltuH"] / max(partH, 1e-8))
                )
                empL_next = (
                    s["empL"]
                    + p["recovery_emp_L"] * (1.0 - s["empL"]) * (s["nL"] / max(partL, 1e-8))
                    - p["scar_emp_L"] * (s["ltuL"] / max(partL, 1e-8))
                )
            else:
                empH_next = 1.0
                empL_next = 1.0

            empH_next = min(max(empH_next, 0.65), 1.05)
            empL_next = min(max(empL_next, 0.45), 1.05)

            # Participation
            if self.labor_hysteresis:
                partH_next = (
                    s["part_H"]
                    + p["reentry_rate"] * (p["mass_H"] * p["part_ss"] - s["part_H"])
                    - p["scar_part_H"] * (s["ltuH"] / max(p["mass_H"], 1e-8))
                    + p["recovery_part_H"] * (nH_next - s["nH"])
                )
                partL_next = (
                    s["part_L"]
                    + p["reentry_rate"] * (p["mass_L"] * p["part_ss"] - s["part_L"])
                    - p["scar_part_L"] * (s["ltuL"] / max(p["mass_L"], 1e-8))
                    + p["recovery_part_L"] * (nL_next - s["nL"])
                )
            else:
                partH_next = p["mass_H"] * p["part_ss"]
                partL_next = p["mass_L"] * p["part_ss"]

            partH_next = min(max(partH_next, 0.22), p["mass_H"] * 0.82)
            partL_next = min(max(partL_next, 0.10), p["mass_L"] * 0.80)

            uH_next = max(0.0, partH_next - nH_next)
            uL_next = max(0.0, partL_next - nL_next)
            ltuH_next = min(ltuH_next, uH_next)
            ltuL_next = min(ltuL_next, uL_next)

        else:
            # Representative labor version
            partH = s["part_H"]
            stuH = max(0.0, s["uH"] - s["ltuH"])
            jfH = p["search_success_H"] * hiring_multiplier * max(s["empH"], 0.6)
            jfH = min(max(jfH, 0.01), 0.90)

            sepH = p["separation_H"] * (1 - 0.5 * x_next)
            sepH = min(max(sepH, 0.005), 0.12)

            hiresH = jfH * (stuH + 0.30 * s["ltuH"])
            sepsH = sepH * s["nH"]

            nH_next = max(0.0, s["nH"] + hiresH - sepsH)
            nL_next = 0.0

            uH_pre = max(0.0, partH - nH_next)
            ltuH_next = (1 - p["exit_ltu"]) * s["ltuH"] + p["lambda_u"] * max(stuH, 0.0)
            ltuH_next = min(ltuH_next, uH_pre)

            if self.labor_hysteresis:
                empH_next = (
                    s["empH"]
                    + p["recovery_emp_H"] * (1.0 - s["empH"]) * (s["nH"] / max(partH, 1e-8))
                    - p["scar_emp_H"] * (s["ltuH"] / max(partH, 1e-8))
                )
                partH_next = (
                    s["part_H"]
                    + p["reentry_rate"] * (p["part_ss"] - s["part_H"])
                    - p["scar_part_H"] * s["ltuH"]
                    + p["recovery_part_H"] * (nH_next - s["nH"])
                )
            else:
                empH_next = 1.0
                partH_next = p["part_ss"]

            empH_next = min(max(empH_next, 0.65), 1.05)
            partH_next = min(max(partH_next, 0.35), 0.80)
            uH_next = max(0.0, partH_next - nH_next)
            ltuH_next = min(ltuH_next, uH_next)

            partL_next = 0.0
            uL_next = 0.0
            ltuL_next = 0.0
            empL_next = 0.0

        # 11) Recompute supply potential using next states
        next_state_temp = {
            **s,
            "K": K_next,
            "Z": Z_next,
            "nH": nH_next,
            "nL": nL_next,
            "empH": empH_next,
            "empL": empL_next,
        }
        y_pot_next = self.potential_output(next_state_temp)

        # 12) Output tied partially to demand and partially supply-constrained
        Y_next = max(0.55, min(Y_next, 1.15 * y_pot_next + 0.10))
        x_next = Y_next / y_pot_next - 1.0

        # 13) Welfare flow
        unemployment_rate = (uH_next + uL_next) / max(partH_next + partL_next, 1e-8)
        welfare_flow = -(
            p["lambda_pi_welfare"] * (pi_next - p["pi_target_q"])**2
            + p["lambda_ygap_welfare"] * (x_next**2)
            + p["lambda_u_welfare"] * ((unemployment_rate - p["u_ss"])**2)
        )

        next_state = {
            "x": x_next,
            "pi": pi_next,
            "i": i_next,
            "rn": rn,
            "d": d_next,
            "Y": Y_next,
            "K": K_next,
            "Z": Z_next,
            "I": I_next,
            "Iz": Iz_next,
            "part_H": partH_next,
            "part_L": partL_next,
            "nH": nH_next,
            "nL": nL_next,
            "uH": uH_next,
            "uL": uL_next,
            "ltuH": ltuH_next,
            "ltuL": ltuL_next,
            "empH": empH_next,
            "empL": empL_next,
            "mc": mc_next,
            "A_eff": (K_next / p["K_ss"])**p["alpha_k"] * (Z_next / p["Z_ss"])**p["alpha_z"],
            "N_eff": self.effective_labor(next_state_temp),
            "Y_pot": y_pot_next,
            "welfare_flow": welfare_flow,
        }
        return next_state

    # --------------------------------------------------------
    # Simulation routine
    # --------------------------------------------------------
    def simulate_irf(self, T=40, shock_size=None):
        if shock_size is None:
            shock_size = self.p["default_demand_shock"]

        ss = self.steady_state()
        states = []
        s = ss.copy()

        for t in range(T):
            innov = shock_size if t == 0 else 0.0
            s = self.step(s, d_innov=innov)
            states.append(s.copy())

        df = pd.DataFrame(states)
        df["t"] = np.arange(1, T + 1)

        # Aggregates
        df["participation"] = df["part_H"] + df["part_L"]
        df["employment"] = df["nH"] + df["nL"]
        df["unemployment"] = df["uH"] + df["uL"]
        df["u_rate"] = df["unemployment"] / df["participation"]
        df["ltu_total"] = df["ltuH"] + df["ltuL"]
        df["ltu_share"] = df["ltu_total"] / np.maximum(df["unemployment"], 1e-8)
        df["output_per_worker"] = df["Y"] / np.maximum(df["employment"], 1e-8)
        df["employment_pop"] = df["employment"]
        df["investment_share"] = df["I"] / df["Y"]
        df["upstream_share"] = df["Iz"] / df["Y"]
        df["welfare_cum"] = df["welfare_flow"].cumsum()

        # Deviations from steady state
        ssY = ss["Y"]
        ssYpot = ss["Y_pot"]
        ssEmp = ss["nH"] + ss["nL"]
        ssPart = ss["part_H"] + ss["part_L"]
        ssUrate = (ss["uH"] + ss["uL"]) / (ss["part_H"] + ss["part_L"])
        ssLTUshare = (ss["ltuH"] + ss["ltuL"]) / max((ss["uH"] + ss["uL"]), 1e-8)
        ssOPW = ssY / max(ssEmp, 1e-8)

        df["Y_pct"] = 100 * (df["Y"] / ssY - 1)
        df["Ypot_pct"] = 100 * (df["Y_pot"] / ssYpot - 1)
        df["employment_pct"] = 100 * (df["employment"] / ssEmp - 1)
        df["part_pp"] = 100 * (df["participation"] - ssPart)
        df["u_rate_pp"] = 100 * (df["u_rate"] - ssUrate)
        df["ltu_share_pp"] = 100 * (df["ltu_share"] - ssLTUshare)
        df["infl_ann_pp"] = 400 * (df["pi"] - self.p["pi_target_q"])
        df["rate_ann_pp"] = 400 * (df["i"] - (self.p["r_star_q"] + self.p["pi_target_q"]))
        df["inv_pct"] = 100 * (df["I"] / self.p["I_ss"] - 1)
        df["iz_pct"] = 100 * (df["Iz"] / self.p["Iz_ss"] - 1)
        df["K_pct"] = 100 * (df["K"] / self.p["K_ss"] - 1)
        df["Z_pct"] = 100 * (df["Z"] / self.p["Z_ss"] - 1)
        df["opw_pct"] = 100 * (df["output_per_worker"] / ssOPW - 1)
        df["empH_pct"] = 100 * (df["empH"] - 1.0)
        df["empL_pct"] = 100 * (df["empL"] - (1.0 if self.heterogeneity else 0.0))

        return df, ss

# ============================================================
# 3. ANALYSIS FUNCTIONS
# ============================================================

def summarize_irf(df, label):
    horizons = [1, 4, 8, 12, 20, 40]
    rows = []
    for h in horizons:
        idx = h - 1
        rows.append({
            "model": label,
            "horizon_q": h,
            "Y_pct": df.loc[idx, "Y_pct"],
            "Ypot_pct": df.loc[idx, "Ypot_pct"],
            "employment_pct": df.loc[idx, "employment_pct"],
            "part_pp": df.loc[idx, "part_pp"],
            "u_rate_pp": df.loc[idx, "u_rate_pp"],
            "ltu_share_pp": df.loc[idx, "ltu_share_pp"],
            "opw_pct": df.loc[idx, "opw_pct"],
            "inv_pct": df.loc[idx, "inv_pct"],
            "iz_pct": df.loc[idx, "iz_pct"],
            "infl_ann_pp": df.loc[idx, "infl_ann_pp"],
            "rate_ann_pp": df.loc[idx, "rate_ann_pp"],
            "cum_welfare": df.loc[idx, "welfare_cum"],
        })
    return pd.DataFrame(rows)

def hysteresis_decomposition(df_full, df_no_labor, df_no_capital):
    """
    Decompose permanent output loss at long horizon into labor and capital channels.
    """
    h = len(df_full) - 1
    full_loss = -df_full.loc[h, "Ypot_pct"]
    no_labor_loss = -df_no_labor.loc[h, "Ypot_pct"]
    no_capital_loss = -df_no_capital.loc[h, "Ypot_pct"]

    labor_contrib = max(0.0, full_loss - no_labor_loss)
    capital_contrib = max(0.0, full_loss - no_capital_loss)
    interaction = max(0.0, full_loss - labor_contrib - capital_contrib)

    out = pd.DataFrame({
        "component": ["full_loss", "labor_channel", "capital_channel", "interaction"],
        "value_pct_potential_output": [full_loss, labor_contrib, capital_contrib, interaction]
    })
    return out

def shock_sensitivity(model, shock_grid, T=40):
    rows = []
    for sh in shock_grid:
        df, _ = model.simulate_irf(T=T, shock_size=sh)
        rows.append({
            "shock_size": sh,
            "peak_output_drop_pct": df["Y_pct"].min(),
            "peak_employment_drop_pct": df["employment_pct"].min(),
            "peak_participation_drop_pp": df["part_pp"].min(),
            "peak_ltu_share_increase_pp": df["ltu_share_pp"].max(),
            "long_run_Ypot_loss_pct": df.iloc[-1]["Ypot_pct"],
            "long_run_employment_loss_pct": df.iloc[-1]["employment_pct"],
            "cum_welfare_40q": df.iloc[-1]["welfare_cum"]
        })
    return pd.DataFrame(rows)

# ============================================================
# 4. PLOTTING
# ============================================================

def save_figure(fig, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)

def plot_core_macro(df, title, filename):
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    ax = axes.ravel()

    ax[0].plot(df["t"], df["Y_pct"])
    ax[0].axhline(0, linestyle="--", linewidth=0.8)
    ax[0].set_title("Output (%)")

    ax[1].plot(df["t"], df["Ypot_pct"])
    ax[1].axhline(0, linestyle="--", linewidth=0.8)
    ax[1].set_title("Potential output (%)")

    ax[2].plot(df["t"], df["infl_ann_pp"])
    ax[2].axhline(0, linestyle="--", linewidth=0.8)
    ax[2].set_title("Inflation deviation (annualized pp)")

    ax[3].plot(df["t"], df["rate_ann_pp"])
    ax[3].axhline(0, linestyle="--", linewidth=0.8)
    ax[3].set_title("Policy rate deviation (annualized pp)")

    ax[4].plot(df["t"], df["inv_pct"])
    ax[4].axhline(0, linestyle="--", linewidth=0.8)
    ax[4].set_title("Physical investment (%)")

    ax[5].plot(df["t"], df["iz_pct"])
    ax[5].axhline(0, linestyle="--", linewidth=0.8)
    ax[5].set_title("Upstream/intangible investment (%)")

    fig.suptitle(title)
    save_figure(fig, filename)

def plot_labor(df, title, filename):
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    ax = axes.ravel()

    ax[0].plot(df["t"], df["employment_pct"])
    ax[0].axhline(0, linestyle="--", linewidth=0.8)
    ax[0].set_title("Employment (%)")

    ax[1].plot(df["t"], df["part_pp"])
    ax[1].axhline(0, linestyle="--", linewidth=0.8)
    ax[1].set_title("Participation (pp)")

    ax[2].plot(df["t"], df["u_rate_pp"])
    ax[2].axhline(0, linestyle="--", linewidth=0.8)
    ax[2].set_title("Unemployment rate (pp)")

    ax[3].plot(df["t"], df["ltu_share_pp"])
    ax[3].axhline(0, linestyle="--", linewidth=0.8)
    ax[3].set_title("LTU share (pp)")

    ax[4].plot(df["t"], df["empH_pct"], label="High type")
    if "empL_pct" in df.columns:
        ax[4].plot(df["t"], df["empL_pct"], label="Low type")
    ax[4].axhline(0, linestyle="--", linewidth=0.8)
    ax[4].set_title("Employability scars (%)")
    ax[4].legend()

    ax[5].plot(df["t"], df["opw_pct"])
    ax[5].axhline(0, linestyle="--", linewidth=0.8)
    ax[5].set_title("Output per worker (%)")

    fig.suptitle(title)
    save_figure(fig, filename)

def plot_model_comparison(dfs, labels, filename):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    ax = axes.ravel()

    for df, lab in zip(dfs, labels):
        ax[0].plot(df["t"], df["Y_pct"], label=lab)
        ax[1].plot(df["t"], df["Ypot_pct"], label=lab)
        ax[2].plot(df["t"], df["employment_pct"], label=lab)
        ax[3].plot(df["t"], df["part_pp"], label=lab)

    titles = ["Output (%)", "Potential output (%)", "Employment (%)", "Participation (pp)"]
    for i in range(4):
        ax[i].axhline(0, linestyle="--", linewidth=0.8)
        ax[i].set_title(titles[i])
        ax[i].legend()

    fig.suptitle("Model comparison")
    save_figure(fig, filename)

def plot_decomposition(decomp_df, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(decomp_df["component"], decomp_df["value_pct_potential_output"])
    ax.set_title("Long-run hysteresis decomposition")
    ax.set_ylabel("Percent of potential output")
    save_figure(fig, filename)

def plot_sensitivity(sens_df, filename):
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    ax = axes.ravel()

    x = 100 * sens_df["shock_size"]

    ax[0].plot(x, sens_df["peak_output_drop_pct"], marker="o")
    ax[0].set_title("Peak output response (%)")

    ax[1].plot(x, sens_df["long_run_Ypot_loss_pct"], marker="o")
    ax[1].set_title("Long-run potential output loss (%)")

    ax[2].plot(x, sens_df["peak_employment_drop_pct"], marker="o")
    ax[2].set_title("Peak employment response (%)")

    ax[3].plot(x, sens_df["peak_ltu_share_increase_pp"], marker="o")
    ax[3].set_title("Peak LTU share increase (pp)")

    for a in ax:
        a.axhline(0, linestyle="--", linewidth=0.8)
        a.set_xlabel("Initial demand shock (%)")

    save_figure(fig, filename)

def plot_scarring_curve(df, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    if "empL_pct" in df.columns:
        ax.plot(df["ltu_share"], df["empL_pct"], label="Low-productivity workers")
        ax.plot(df["ltu_share"], df["empH_pct"], label="High-productivity workers")
    else:
        ax.plot(df["ltu_share"], df["empH_pct"], label="Representative worker")
    ax.set_title("Internal labor scarring curve")
    ax.set_xlabel("Long-term unemployment share")
    ax.set_ylabel("Employability deviation (%)")
    ax.legend()
    save_figure(fig, filename)

# ============================================================
# 5. TABLE EXPORTS
# ============================================================

def calibration_table(pars):
    rows = [{"parameter": k, "value": v} for k, v in pars.items()]
    return pd.DataFrame(rows)

# ============================================================
# 6. MAIN EXECUTION
# ============================================================

def main():
    pars = get_us_calibration()

    # Main model variants
    model_full = NKHysteresisModel(
        pars,
        labor_hysteresis=True,
        capital_hysteresis=True,
        heterogeneity=True,
        name="full"
    )

    model_no_labor = NKHysteresisModel(
        pars,
        labor_hysteresis=False,
        capital_hysteresis=True,
        heterogeneity=True,
        name="no_labor_hysteresis"
    )

    model_no_capital = NKHysteresisModel(
        pars,
        labor_hysteresis=True,
        capital_hysteresis=False,
        heterogeneity=True,
        name="no_capital_hysteresis"
    )

    model_rep = NKHysteresisModel(
        pars,
        labor_hysteresis=True,
        capital_hysteresis=True,
        heterogeneity=False,
        name="representative_labor"
    )

    # Simulations
    df_full, ss_full = model_full.simulate_irf(T=40)
    df_no_labor, _ = model_no_labor.simulate_irf(T=40)
    df_no_capital, _ = model_no_capital.simulate_irf(T=40)
    df_rep, _ = model_rep.simulate_irf(T=40)

    # Summaries
    table_calib = calibration_table(pars)

    table_irf = pd.concat([
        summarize_irf(df_full, "full"),
        summarize_irf(df_no_labor, "no_labor_hysteresis"),
        summarize_irf(df_no_capital, "no_capital_hysteresis"),
        summarize_irf(df_rep, "representative_labor"),
    ], ignore_index=True)

    table_decomp = hysteresis_decomposition(df_full, df_no_labor, df_no_capital)

    shock_grid = np.array([-0.01, -0.02, -0.03, -0.04, -0.05, -0.06])
    table_sens = shock_sensitivity(model_full, shock_grid=shock_grid, T=40)

    # Export CSV
    table_calib.to_csv(os.path.join(OUTPUT_DIR, "table_1_calibration.csv"), index=False)
    table_irf.to_csv(os.path.join(OUTPUT_DIR, "table_2_irf_summary.csv"), index=False)
    table_decomp.to_csv(os.path.join(OUTPUT_DIR, "table_3_hysteresis_decomposition.csv"), index=False)
    table_sens.to_csv(os.path.join(OUTPUT_DIR, "table_4_shock_sensitivity.csv"), index=False)

    # Export model trajectories
    with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "nk_hysteresis_us_outputs.xlsx"), engine="xlsxwriter") as writer:
        df_full.to_excel(writer, sheet_name="full_model", index=False)
        df_no_labor.to_excel(writer, sheet_name="no_labor_hysteresis", index=False)
        df_no_capital.to_excel(writer, sheet_name="no_capital_hysteresis", index=False)
        df_rep.to_excel(writer, sheet_name="representative_labor", index=False)
        table_calib.to_excel(writer, sheet_name="calibration", index=False)
        table_irf.to_excel(writer, sheet_name="irf_summary", index=False)
        table_decomp.to_excel(writer, sheet_name="decomposition", index=False)
        table_sens.to_excel(writer, sheet_name="sensitivity", index=False)

    # Figures
    plot_core_macro(df_full, "Figure 1. Core macro IRFs", "figure_1_core_macro_irfs.png")
    plot_labor(df_full, "Figure 2. Labor market and productivity IRFs", "figure_2_labor_irfs.png")
    plot_model_comparison(
        [df_full, df_no_labor, df_no_capital, df_rep],
        ["Full", "No labor hysteresis", "No capital hysteresis", "Representative labor"],
        "figure_3_model_comparison.png"
    )
    plot_decomposition(table_decomp, "figure_4_decomposition.png")
    plot_sensitivity(table_sens, "figure_5_sensitivity.png")
    plot_scarring_curve(df_full, "figure_6_labor_scarring_curve.png")

    # README
    with open(os.path.join(OUTPUT_DIR, "README_outputs.txt"), "w", encoding="utf-8") as f:
        f.write("NK Hysteresis Model Outputs\n")
        f.write("===========================\n\n")
        f.write("Generated files:\n")
        f.write("- table_1_calibration.csv\n")
        f.write("- table_2_irf_summary.csv\n")
        f.write("- table_3_hysteresis_decomposition.csv\n")
        f.write("- table_4_shock_sensitivity.csv\n")
        f.write("- nk_hysteresis_us_outputs.xlsx\n")
        f.write("- figure_1_core_macro_irfs.png\n")
        f.write("- figure_2_labor_irfs.png\n")
        f.write("- figure_3_model_comparison.png\n")
        f.write("- figure_4_decomposition.png\n")
        f.write("- figure_5_sensitivity.png\n")
        f.write("- figure_6_labor_scarring_curve.png\n\n")

        f.write("Editorial interpretation:\n")
        f.write("- The full model should show persistent output and employment losses.\n")
        f.write("- Participation and LTU are the main labor-side hysteresis channels.\n")
        f.write("- Capital/upstream capacity reinforces supply-side persistence.\n")
        f.write("- Output per worker should move less than employment, matching the intended fact pattern.\n")

    # Print concise summary to notebook / Colab
    print("\nDONE. Main files saved in:", OUTPUT_DIR)
    print("\nLong-run full-model results (quarter 40):")
    print(df_full.loc[df_full.index[-1], [
        "Y_pct", "Ypot_pct", "employment_pct", "part_pp",
        "u_rate_pp", "ltu_share_pp", "opw_pct", "inv_pct", "iz_pct", "welfare_cum"
    ]])

    print("\nHysteresis decomposition:")
    print(table_decomp)

    print("\nShock sensitivity:")
    print(table_sens)

# ============================================================
# 7. RUN
# ============================================================

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.2 MB/s eta 0:00:00

DONE. Main files saved in: nk_hysteresis_outputs

Long-run full-model results (quarter 40):
Y_pct              2.812013
Ypot_pct           2.320419
employment_pct    -6.064194
part_pp           -1.052086
u_rate_pp          4.247192
ltu_share_pp      34.184349
opw_pct            9.449227
inv_pct            0.828935
iz_pct             1.003347
welfare_cum       -0.868537
Name: 39, dtype: float64

Hysteresis decomposition:
         component  value_pct_potential_output
0        full_loss                   -2.320419
1    labor_channel                    2.665273
2  capital_channel                    0.000000
3      interaction                    0.000000

Shock sensitivity:
   shock_size  peak_output_drop_pct  peak_employment_drop_pct  \
0       -0.01              3.176603                 -6.028456   
1       -0.02              2.708743                 -6.040465   
2       -0.03              0.669299          